In [ ]:
!pip install -q langchain langchain-community langchain-google-genai pypdf chromadb reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_filename = "Preguntas_Frecuentes_Mercado_Central_24h.pdf"
doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
styles = getSampleStyleSheet()

content = []

title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=16, leading=20, spaceAfter=12)
heading_style = ParagraphStyle('HeadingStyle', parent=styles['Heading2'], fontSize=12, leading=16, spaceAfter=8)
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=10, leading=14, spaceAfter=8)

content.append(Paragraph("<b>Mercado Central 24h — Preguntas Frecuentes (FAQ) y Políticas</b>", title_style))
content.append(Spacer(1, 10))

content.append(Paragraph("<b>1. Horarios y Entregas</b>", heading_style))
content.append(Paragraph("Mercado Central 24h opera de forma continua las 24 horas del día, los 7 días de la semana en sus tiendas físicas. Los servicios de delivery y entregas a domicilio programadas se realizan entre las 06:00 y las 22:00 horas.", body_style))

content.append(Paragraph("<b>2. Programa Cliente VIP Central</b>", heading_style))
content.append(Paragraph("El programa Cliente VIP Central permite acumular 1 punto por cada $10 MXN de compra. Los puntos acumulados se pueden canjear por envíos gratuitos y descuentos en compras futuras. La inscripción es gratuita desde la aplicación móvil.", body_style))

content.append(Paragraph("<b>3. Política de Devoluciones y Reembolsos</b>", heading_style))
content.append(Paragraph("Para productos no perecederos, el cliente cuenta con hasta 30 días para solicitar cambio o devolución presentando el ticket de compra. Para productos perecederos y frescos, las devoluciones se deben reportar dentro de las primeras 24 horas posteriores a la compra.", body_style))

content.append(Paragraph("<b>4. Métodos de Pago Aceptados</b>", heading_style))
content.append(Paragraph("Aceptamos tarjetas de crédito y débito (Visa, Mastercard, American Express), pagos sin contacto, transferencias electrónicas y efectivo tanto en caja física como en entregas contra entrega.", body_style))

doc.build(content)
print("✅ Archivo PDF creado con éxito.")

✅ Archivo PDF creado con éxito.


In [ ]:
!pip install -q -U langchain-classic langchain-text-splitters langchain-chroma langchain-google-genai pypdf

In [ ]:
!pip install -q fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 12.4 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. Cargar API Key para Gemini
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    api_key = "AQ.Ab8RN6LGyvzkWMLC4GOoVlrcuGDqrtX0HHn05FWwxABPw5_TFg"

os.environ["GOOGLE_API_KEY"] = api_key

pdf_path = "Preguntas_Frecuentes_Mercado_Central_24h.pdf"

if os.path.exists(pdf_path):
    # 2. Cargar PDF
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # 3. Fragmentación (Chunking)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)

    # 4. Embeddings locales (Rápidos, 100% compatibles, sin error de API)
    embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 5. Generador Gemini para respuestas
    try:
        llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
    except Exception:
        llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

    # 6. Prompt RAG
    system_prompt = (
        "Eres el asistente virtual oficial de Mercado Central 24h.\n"
        "Responde a las preguntas utilizando únicamente la siguiente información de contexto:\n"
        "{context}\n\n"
        "Si la respuesta no se encuentra en el contexto, di amablemente que no dispones de esa información."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    print("🚀 ¡Agente de Mercado Central 24h cargado y listo!")
else:
    print("❌ No se encontró el archivo PDF.")

🚀 ¡Agente de Mercado Central 24h cargado y listo!


In [ ]:
# Definir preguntas de prueba sobre las políticas de Mercado Central 24h
preguntas = [
    "¿Cómo funciona el programa Cliente VIP Central?",
    "¿Cuál es el plazo para devolver un producto perecedero?",
    "¿Cuáles son los horarios del servicio de delivery?"
]

for idx, p in enumerate(preguntas, 1):
    respuesta = rag_chain.invoke({"input": p})
    print(f"❓ Pregunta {idx}: {p}")
    print(f"🤖 Respuesta: {respuesta['answer']}")
    print("-" * 60)

❓ Pregunta 1: ¿Cómo funciona el programa Cliente VIP Central?
🤖 Respuesta: El programa **Cliente VIP Central** funciona de la siguiente manera:

* **Acumulación de puntos:** Acumulas 1 punto por cada $10 MXN de compra.
* **Beneficios:** Los puntos que acumules los puedes canjear por envíos gratuitos y descuentos en tus compras futuras.
* **Inscripción:** Es totalmente gratuita y puedes registrarte directamente desde nuestra aplicación móvil.
------------------------------------------------------------
❓ Pregunta 2: ¿Cuál es el plazo para devolver un producto perecedero?
🤖 Respuesta: Para los productos perecederos y frescos, las devoluciones se deben reportar dentro de las primeras **24 horas** posteriores a la compra.
------------------------------------------------------------
❓ Pregunta 3: ¿Cuáles son los horarios del servicio de delivery?
🤖 Respuesta: Los servicios de delivery y entregas a domicilio programadas se realizan entre las **06:00 y las 22:00 horas**.
---------------------